<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/01-learning-from-data.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 1 — What Learning From Data Means

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/what-learning-from-data-means/).

Everything here happens in a world we wrote ourselves, so we always know the right answer and can check every model against it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def mae(predicted, actual):
    """Mean absolute error: on average, how many minutes out."""
    return np.mean(np.abs(predicted - actual))

## 1. A world where we know the answer

In [ ]:
def bicycle(km):
    """The true delivery time in minutes. In real life this formula is unknown."""
    return 10 + 3.2 * km + 0.15 * km**2

def make_orders(n, world=bicycle, lo=0.5, hi=8.0, noise=4.0, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(lo, hi, n)
    minutes = world(x) + rng.normal(0, noise, n)   # traffic, lifts, a slow kitchen
    return x, minutes

km_train, min_train = make_orders(200, seed=1)    # last month
km_test, min_test = make_orders(1000, seed=2)     # next month's orders

print(f'{len(km_train)} orders to learn from, {len(km_test)} new ones to predict')
print('first five:', np.round(km_train[:5], 1), 'km')
print('     took :', np.round(min_train[:5], 1), 'min')

## 2. The memoriser

In [ ]:
def knn_predict(x_known, y_known, x_new, k):
    """Predict each new order as the average of the k most similar known orders."""
    gaps = np.abs(x_new[:, None] - x_known[None, :])   # every new order against every known one
    nearest = np.argsort(gaps, axis=1)[:, :k]
    return y_known[nearest].mean(axis=1)

seen = knn_predict(km_train, min_train, km_train, k=1)
new = knn_predict(km_train, min_train, km_test, k=1)
print(f'k=1 on orders it has seen    : {mae(seen, min_train):.2f} min')
print(f'k=1 on orders it has not seen: {mae(new, min_test):.2f} min')

## 3. Averaging more neighbours

In [ ]:
print(f'{"k":>5} {"seen":>6} {"new":>6}')
for k in [1, 5, 25, 100, 200]:
    seen = mae(knn_predict(km_train, min_train, km_train, k), min_train)
    new = mae(knn_predict(km_train, min_train, km_test, k), min_test)
    print(f'{k:>5} {seen:>6.2f} {new:>6.2f}')

At `k=200` every prediction averages all 200 orders, so the model says the same number for a 1 km trip and an 8 km one.

## 4. Nothing gets under the noise

In [ ]:
always_average = np.full(len(min_test), min_train.mean())
print(f'the true formula, on new orders: {mae(bicycle(km_test), min_test):.2f} min')
print(f'always guess the average       : {mae(always_average, min_test):.2f} min')

## 5. Family, loss, search: a line by brute force

In [ ]:
slopes = np.arange(0, 8.001, 0.05)        # the family: every line with one of these slopes
intercepts = np.arange(0, 30.001, 0.1)    # and one of these intercepts

best_loss = np.inf
for a in slopes:                           # the search: try every one of them
    for b in intercepts:
        loss = np.mean((a * km_train + b - min_train) ** 2)   # the loss: mean squared error
        if loss < best_loss:
            best_a, best_b, best_loss = a, b, loss

print(f'tried {len(slopes) * len(intercepts):,} lines')
print(f'best: minutes = {best_b:.1f} + {best_a:.2f} x km')
print(f'on new orders: {mae(best_a * km_test + best_b, min_test):.2f} min')

## 6. The one line you would ship

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

X_train, X_test = km_train.reshape(-1, 1), km_test.reshape(-1, 1)   # scikit-learn wants a table

line = LinearRegression().fit(X_train, min_train)
knn = KNeighborsRegressor(n_neighbors=25).fit(X_train, min_train)

print(f'LinearRegression: minutes = {line.intercept_:.1f} + {line.coef_[0]:.2f} x km, '
      f'{mae(line.predict(X_test), min_test):.2f} min on new orders')
print('KNeighborsRegressor agrees with ours:',
      np.allclose(knn.predict(X_test), knn_predict(km_train, min_train, km_test, 25)))

In [ ]:
grid = np.linspace(0.5, 8, 400)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(km_train, min_train, s=12, color='grey', alpha=0.6, label='last month')
ax.plot(grid, knn_predict(km_train, min_train, grid, 1), lw=1, label='k=1, memorised')
ax.plot(grid, knn_predict(km_train, min_train, grid, 25), lw=2, label='k=25')
ax.plot(grid, line.predict(grid.reshape(-1, 1)), lw=2, ls='--', label='straight line')
ax.set_xlabel('distance (km)')
ax.set_ylabel('minutes')
ax.legend()
plt.show()

## 7. No algorithm wins everywhere

In [ ]:
# World A: the same distance problem, with only 20 orders to learn from
km_few, min_few = make_orders(20, seed=3)
few_line = LinearRegression().fit(km_few.reshape(-1, 1), min_few)
few_knn = KNeighborsRegressor(n_neighbors=10).fit(km_few.reshape(-1, 1), min_few)

# World B: time of day, with a lunch rush and a dinner rush
def rush(hour):
    return 25 + 12 * np.exp(-(hour - 13)**2 / 2) + 15 * np.exp(-(hour - 20)**2 / 2)

hr_train, hmin_train = make_orders(200, world=rush, lo=10, hi=23, seed=4)
hr_test, hmin_test = make_orders(1000, world=rush, lo=10, hi=23, seed=5)
H_train, H_test = hr_train.reshape(-1, 1), hr_test.reshape(-1, 1)
rush_line = LinearRegression().fit(H_train, hmin_train)
rush_knn = KNeighborsRegressor(n_neighbors=25).fit(H_train, hmin_train)

print(f'{"":<26} {"line":>6} {"k-NN":>6}')
print(f'{"distance, 20 orders":<26} {mae(few_line.predict(X_test), min_test):>6.2f}'
      f' {mae(few_knn.predict(X_test), min_test):>6.2f}')
print(f'{"time of day, 200 orders":<26} {mae(rush_line.predict(H_test), hmin_test):>6.2f}'
      f' {mae(rush_knn.predict(H_test), hmin_test):>6.2f}')

## 8. The assumption, broken twice

### Longer trips

In [ ]:
km_far, min_far = make_orders(1000, lo=8, hi=16, seed=6)   # the service area doubles
X_far = km_far.reshape(-1, 1)
print(f'line on 8-16 km trips: {mae(line.predict(X_far), min_far):.2f} min')
print(f'k=25 on 8-16 km trips: {mae(knn.predict(X_far), min_far):.2f} min')
print()
for d in [4, 8, 12, 16]:
    print(f'{d:>2} km   truth {bicycle(d):5.1f}   line {line.predict([[d]])[0]:5.1f}'
          f'   k-NN {knn.predict([[d]])[0]:5.1f}')

### Scooters

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

def scooter(km):
    """The fleet switches from bicycles to e-scooters. The model is never retrained."""
    return 10 + 2.2 * km + 0.08 * km**2

cv = cross_val_score(LinearRegression(), X_train, min_train,
                     cv=KFold(5, shuffle=True, random_state=0),
                     scoring='neg_mean_absolute_error')
km_now, min_now = make_orders(1000, world=scooter, seed=7)
after = line.predict(km_now.reshape(-1, 1))

print(f'cross-validated on last month: {-cv.mean():.2f} ± {cv.std():.2f} min')
print(f'deployed after the switch    : {mae(after, min_now):.2f} min')
print(f'average over-estimate        : {np.mean(after - min_now):+.2f} min')

## Try it yourself

1. Set `noise=0` in `make_orders` and rerun sections 1 to 3. Which `k` is best now, and why has the memoriser stopped being a problem?
2. Give World A 2,000 orders instead of 20. How close does k-NN get to the line?
3. In section 8, add 50 scooter orders to the training data and refit the line. How much of the error goes away, and how would you have known to do it?